###page5

In [1]:
def split_text(text, chunk_size=50, overlap=10):
    # 1. 무한 루프 방지: chunk_size는 overlap보다 커야 함
    if chunk_size <= overlap:
        raise ValueError("chunk_size는 overlap보다 커야 합니다.")
       
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
       
        # 2. 텍스트의 끝까지 모두 잘랐다면 불필요한 반복 종료
        if end >= len(text):
            break
           
        start = end - overlap
       
    return chunks


page6

In [4]:
long_text = (
    "우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. "
    "강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 "
    "수강한 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 "
    "이상을 수강하면 수료증을 신청할 수 있습니다."
)

# [1번 문제]
# 위 slide의 split_text 함수를 사용해 long_text를 chunk_size=50,
# overlap=10 으로 나누고, 청크 개수를 출력하세요.
test_vals = split_text(long_text)
print(len(test_vals))

# [2번 문제]
# 나눠진 청크를 순서대로 하나씩 출력해, 앞뒤 청크가 어떻게
# 겹치는지(overlap) 확인하세요.
for val in test_vals:
    print(f"{val}\n")

# [도전 문제]
# chunk_size=80, overlap=0 으로 다시 나눠보고, 청크 개수와
# 내용이 어떻게 달라지는지 비교하세요.
test_vals = split_text(long_text,80,0)
print(len(test_vals))

for val in test_vals:
    print(f"{val}\n")
# 여기에 코드를 작성해보세요


4
우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. 강의 시작 후 7일

강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 수강한 강의 비율에 따라

 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 이상을 수강하면 수료증

을 수강하면 수료증을 신청할 수 있습니다.

2
우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. 강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 수강한

 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 이상을 수강하면 수료증을 신청할 수 있습니다.



page7

In [6]:
from langchain_openai import OpenAIEmbeddings

from dotenv import load_dotenv
import os # 기본 내장

load_dotenv()  # 상위 폴더로 올라가면서 찾아줌 .env 내용을 환경변수로 등록
api_key = os.getenv("OPENAI_API_KEY")  # 환경변수에서 꺼내쓰기

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key #.env에서 불러온 키 사용
)

vector = embeddings.embed_query("안녕하세요")
print(len(vector))  # 1536
print(vector)

1536
[-0.002506256103515625, -0.061248779296875, -0.0084381103515625, 0.03155517578125, 0.0311279296875, -0.04986572265625, -0.05914306640625, 0.03240966796875, -0.01409912109375, -0.061431884765625, -0.020721435546875, 0.00679779052734375, 0.009185791015625, -0.02008056640625, -0.01149749755859375, 0.035186767578125, -0.04656982421875, 0.0039825439453125, -0.0141143798828125, 0.0273590087890625, 0.057098388671875, -0.017242431640625, -0.03131103515625, -0.0202178955078125, 0.0440673828125, 0.062408447265625, 0.056884765625, 0.0005273818969726562, 0.0171356201171875, -0.05352783203125, 0.031402587890625, -0.027618408203125, -0.0174713134765625, 0.0012063980102539062, -0.0023593902587890625, 0.025390625, 0.02264404296875, -0.04388427734375, -0.011749267578125, -0.034210205078125, -0.0135345458984375, -0.0182647705078125, -0.0030384063720703125, 0.047821044921875, 0.052520751953125, 0.01273345947265625, -0.044036865234375, 0.0281982421875, 0.0438232421875, 0.037078857421875, -0.024276733

### page8.chapter2.벡터저장소와 유사도 검색

In [11]:
from langchain_core.vectorstores import (
    InMemoryVectorStore,
)

# 청크 처리된 데이터
docs = [
    "우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. ",
    "강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 ",
    "수강한 강의 비율에 따라 부분 환불이 진행됩니다. 전체 강의의 80퍼센트 ",
    "이상을 수강하면 수료증을 신청할 수 있습니다."
]

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key #.env에서 불러온 키 사용
)

# 벡터 스토어에 문서와 임베딩 모델을 이용해서 벡터로 변환해서 저장
vectorstore = InMemoryVectorStore.from_texts(
    docs, embedding=embeddings
)
# 질문과 유사도가 높은 벡터 데이터를 2개 찾아서 반환해줌
result=vectorstore.similarity_search("환불 가능 기간은?", k=2)
# 출력해보니: result=[Document, Document] -> page_content속성에 내용이 저장되어 있음
print(result)
for doc in result:
    print(doc.page_content)



[Document(id='9f66a3ea-88fc-4c7a-90a0-ea6b7a6e7ebb', metadata={}, page_content='강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 '), Document(id='fa59754a-4b61-4bc3-abff-591eb719775b', metadata={}, page_content='우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. ')]
강의 시작 후 7일 이내에는 전액 환불이 가능하며, 7일이 지나면 
우리 온라인 강의 서비스는 결제일로부터 180일간 수강할 수 있습니다. 


page9.chapter2.[실습]임베딩+벡터저장소 만들기

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

docs = [
    "환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.",
    "수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.",
    "수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.",
]

# [1번 문제]
# OpenAIEmbeddings()로 embeddings를 만들고, InMemoryVectorStore.from_texts로
# docs를 저장한 vectorstore를 만드세요.
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key #.env에서 불러온 키 사용
)

vectorstore = InMemoryVectorStore.from_texts(
    docs, embedding=embeddings
)
# [2번 문제]
# vectorstore.similarity_search("환불 받고 싶어요", k=2) 결과를 출력하세요.
result = vectorstore.similarity_search("환불 받고 싶어요", k=2)
print(result[0].page_content,result[1].page_content)

# [도전 문제]
# k값을 1과 3으로 각각 바꿔서 검색해보고, 결과 문서 개수가
# 어떻게 달라지는지 확인하세요.
result2 = vectorstore.similarity_search("환불 받고 싶어요", k=3)
print(len(result2))

# 여기에 코드를 작성해보세요


환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다. 수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.
3


### page10

In [14]:
# vectorstore를 langchain에서 사용할 수 있게 retriever 객체로 변환
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2} # 검색 내용 2개 찾기 설정
)

docs = retriever.invoke("환불 규정이 궁금해요")
for d in docs:
    print(d.page_content)


환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.
수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.


### page 11

In [15]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini", #모델 이름
    temperature=0,       #창의적인 답변 정도
    max_tokens=200,      #답변 토큰량
    api_key=api_key,     #api key
)

docs = [
    "환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.",
    "수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.",
    "수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.",
    "수강 과목:AI활용 웹서비스 개발자과정입니다.",
    "수강생: 김형준, 김태경, 조성우, 임호현"
]

embeddings = OpenAIEmbeddings(
    model = "text-embedding-3-small"
)

vectorstore = InMemoryVectorStore.from_texts(
    docs, embedding=embeddings
)

retriever = vectorstore.as_retriever(search_kwargs={"k":2})

def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "문서를 참고해 답하세요. 모르면 모른다고"
     " 답하세요.\n\n문서:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

question = "수강생들의 이름을 알려줘"
result = rag_chain.invoke(question)
print("질문:",question)
print("답변:",result)

질문: 수강생들의 이름을 알려줘
답변: 수강생들의 이름은 김형준, 김태경, 조성우, 임호현입니다.


page11_2.실제 파일 데이터로 처리해보기

In [25]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 1. API 키 로드
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
# 2. LLM 설정
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=300,
    api_key=api_key,
)
# 3. testdocs.txt 파일 읽기 및 청킹(Chunking)
with open("testdocs.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# 텍스트 분할기 설정 (줄바꿈 및 공백 기준으로 적절한 크기로 분할)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,       # 청크 1개의 최대 글자수
    chunk_overlap=30,      # 앞뒤 청크 간 겹치는 글자수
    separators=["\n", " ", ""]
)

docs = text_splitter.split_text(raw_text)
# (참고: 앞선 실습에서 만든 split_text 함수를 사용하셔도 됩니다)
# docs = split_text(raw_text, chunk_size=150, overlap=20)
print(f"총 분할된 청크 수: {len(docs)}개")
print(docs[0])
# 4. 임베딩 및 벡터 저장소 생성
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=api_key
)
vectorstore = InMemoryVectorStore.from_texts(
    docs, embedding=embeddings
)
# 5. 검색기(Retriever) 설정 (유사 문서 3개 검색)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

# 6. 프롬프트 템플릿
prompt = ChatPromptTemplate.from_messages([
    ("system", "문서를 참고해 사실에 기반하여 답하세요.문서에 없는 내용은 모른다고 답하세요.\n\n문서:\n{context}"),
    ("human", "{question}"),
])

# 7. RAG 체인 연결
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
# 8. 질문 및 답변 테스트 (testdocs.txt 내용 기반 질문)
question = "전체 3등인 학생의 이름과 작성 글자수, 평가 코멘트를 알려줘"
result = rag_chain.invoke(question)
print("질문:", question)
print("답변:\n", result)

총 분할된 청크 수: 6개
순위	이름	제출 파일수	인정 일수 (총 23일)	총 작성 글자수	성실도 점수 (10점)	성실도 등급	교강사 오답노트 진단 코멘트
1	김태경	22개	22일	84,375자	10.00점	최우수 S	8.4만 자로 전체 1위 작성량 기록! 깊이 있는 코드 분석과 디버깅 과정 기술.
2	윤제정	1개	    22일	64,355자	10.00점	최우수 S	6.4만 자로 전체 2위 작성량! 단일 파일에 전 회차 꼼꼼히 누적 정리하여 SSR 7.2점 대반등.
질문: 전체 3등인 학생의 이름과 작성 글자수, 평가 코멘트를 알려줘
답변:
 전체 3등인 학생은 지승현이며, 작성 글자수는 39,125자입니다. 평가 코멘트는 "오답노트 22일 3.9만 자 성실도 10점 만점! 프로젝트 협업 시 높은 책임감 기대."입니다.


### page12

In [17]:
# 앞 슬라이드에서 만든 vectorstore를 이어서 사용합니다.

# [1번 문제]
# vectorstore.as_retriever(search_kwargs={"k": 2}) 로 retriever를 만드세요.
retriever = vectorstore.as_retriever(search_kwargs={"k":2})
# [2번 문제]
# retriever.invoke("수료증은 언제 받을 수 있나요?") 의 결과를
# 하나씩 출력하며 어떤 문서가 검색되는지 확인하세요.
docs = retriever.invoke("수료증은 언제 받을 수 있나요?")
for doc in docs:
    print(doc.page_content)
# [도전 문제]
# 문서 내용과 전혀 관련 없는 질문(예: "오늘 날씨 어때?")으로
# retriever.invoke()를 호출해보고, 그래도 문서가 검색되는지,
# 검색된 문서가 실제로 관련 있어 보이는지 살펴보세요.
docs = retriever.invoke("오늘 날씨 어때?")
for doc in docs:
    print(doc.page_content)
# 여기에 코드를 작성해보세요


수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.
수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.
수강생: 김형준, 김태경, 조성우, 임호현
환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.


page15

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# [1번 문제]
# 문서 내용을 하나의 문자열로 합쳐주는 format_docs(docs) 함수를 작성하세요.
def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

# [2번 문제]
# system 메시지에 "문서를 참고해 답하세요. 문서에 없으면 모른다고
# 답하세요.\n\n문서:\n{context}", human 메시지 "{question}" 으로 prompt를 만들고,
# {"context": retriever | format_docs, "question": RunnablePassthrough()}
# | prompt | llm | StrOutputParser() 로 rag_chain을 완성하세요.
prompt = ChatPromptTemplate.from_messages([
    ("system", "문서를 참고해 답하세요. 문서에 없으면 모른다고"
     " 답하세요.\n\n문서:\n{context}"),
    ("human", "{question}"),
])

rag_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)
# [도전 문제]
# rag_chain.invoke("환불 규정이 어떻게 되나요?") 를 실행해 답변을 확인하고,
# 문서에 전혀 없는 질문도 하나 만들어 invoke() 해서 어떻게 답하는지 보세요.
results = rag_chain.invoke("오늘 날씨는 어떻게 되니?")
print("답변:",results)
# 여기에 코드를 작성해보세요


### page17.[종합실습]미니 사내문서Q&A봇

In [ ]:
import os
from dotenv import load_dotenv

# API 키 불러오기 및 필수 라이브러리 임포트
load_dotenv()

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.vectorstores import InMemoryVectorStore

# LLM 및 임베딩 모델 준비
llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    "환불 정책: 강의 시작 후 7일 이내에는 전액 환불이 가능합니다.",
    "수강 기간: 강의는 결제일로부터 180일간 수강할 수 있습니다.",
    "수료증 발급: 전체 강의의 80% 이상을 수강하면 신청할 수 있습니다.",
    "질문 답변: 강의 내용 질문은 Q&A 게시판에서 48시간 이내 답변드립니다.",
]

# [1번 문제] docs로 vectorstore를 만들고, as_retriever(k=2)로
# retriever를 만든 뒤, 이번 시간에 만든 rag_chain 구성 방식대로
# qa_chain을 완성하세요.
vectorstore = InMemoryVectorStore.from_texts(docs, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

def format_docs(docs):
    return "\n".join(d.page_content for d in docs)

prompt = ChatPromptTemplate.from_messages([
    ("system", "문서를 참고해 답하세요. 문서에 없으면 모른다고"
     " 답하세요.\n\n문서:\n{context}"),
    ("human", "{question}"),
])

qa_chain = (
    {"context": retriever | format_docs,
     "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

# [2번 문제] question(문자열)을 받아 qa_chain.invoke(question)을
# 호출해 답변을 반환하는 answer(question) 함수를 작성하세요.
def answer(question):
    return qa_chain.invoke(question)

# [도전 문제] "수강 기간이 얼마나 되나요?" 처럼 문서에 있는 질문과,
# "환불 담당자 전화번호가 뭔가요?" 처럼 문서에 없는 질문을 각각 answer()로
# 실행해, 두 경우에 답변이 어떻게 다른지 비교하세요.
results = answer("수강 기간이 얼마나 되나요?")
print("답변:",results)
results = answer("환불 담당자 전화번호가 뭔가요?")
print("답변:",results)
